# Section 1: Information about the submission

## 1.1 Name and number of the assignment

**Assignment:** Hallucination Detection in Tool Calling

## 1.2 Student name

**Students:** Iman Chantieva, Irina Brodskaya

## 1.3 Additional comments

To reproduce the full pipeline, use the project repository:

<https://github.com/hhfawn/halu_detection>
<https://huggingface.co/Fawnnn/ModernBERT-toolace-hallu-detect>
<https://huggingface.co/datasets/Fawnnn/Hallucination>

The submitted notebook implements a full end-to-end pipeline: dataset loading or generation, synthetic hallucination injection, baseline evaluation, fine-tuning, and final comparison. The notebook is designed so that if the dataset folder is empty or incomplete, it can automatically generate the required JSONL datasets before evaluation or training.

# 2. Technical Report

## 2.1 Methodology

### Problem definition

The goal of this project is to detect hallucinated spans in assistant answers produced in a tool-calling setting. In this setting, the assistant receives a user query and one or more tool outputs. A faithful answer should only contain information supported by the tool output and should not introduce unsupported facts or actions. We formulate the task as a span-detection problem: for each answer, the model must identify the exact character range corresponding to hallucinated content.

The project focuses on three hallucination categories:

1. **Contradictory factual hallucination**: a short factual span in the answer is replaced with another value that contradicts the tool output.
2. **Overgeneration**: the answer contains an additional plausible sentence that is topically related but unsupported by the tool output.
3. **Missing-tool violation**: the answer suggests an action that would require a tool not available in the provided tool list, such as booking, ordering, emailing, or scheduling.

This design allows the benchmark to test both standard factual grounding and tool-specific violations.

### Dataset source and conversion

The source dataset is **ToolACE**, which contains tool-use conversations. Each record is parsed into four components:

- `query`: the user request;
- `context`: the tool output, treated as the ground-truth evidence;
- `output`: the assistant answer;
- `hallucination_labels`: a list of hallucinated character spans.

The parser extracts the first user turn, the last tool or observation turn, and the final assistant answer after the tool call. Records without a valid user query, tool output, or final assistant answer are discarded. This produces examples suitable for RAGTruth-style span-level hallucination detection.

### Automatic dataset generation

The notebook contains an automatic dataset entry point, `ensure_datasets()`. This function checks whether the expected JSONL files exist and contain rows. If the files are already present, they are loaded directly. If the folder is empty, incomplete, or broken, the function calls `generate_datasets()` to create fresh hallucination datasets.

The three expected dataset files are:

| Dataset type | File |
|---|---|
| Contradictory factual hallucination | `toolace_halu_contradiction.jsonl` |
| Overgeneration | `toolace_halu_overgeneration.jsonl` |
| Missing-tool violation | `toolace_halu_missing_tool.jsonl` |

This makes the notebook reproducible after kernel restarts and portable across environments where the dataset folder may initially be empty.

### Synthetic hallucination generation

Synthetic hallucinations are generated using a guided JSON generation setup. For contradiction examples, the generator receives the tool output and the original answer, selects an exact factual substring from the answer, and proposes a contradictory replacement. The replacement is applied only if the original span appears verbatim in the answer. The resulting hallucinated span is recorded as a character range.

For overgeneration and missing-tool violations, the generator appends one additional sentence to the original answer. The appended sentence becomes the labeled hallucinated span. For overgeneration, the sentence introduces unsupported information not found in the tool output. For missing-tool examples, the sentence offers an action requiring a tool that is not available in the tool list.

The generated datasets used in the final run contain:

| Dataset type | Positive examples |
|---|---:|
| Contradictory factual hallucination | 736 |
| Overgeneration | 788 |
| Missing-tool violation | 788 |

For example-level evaluation, each positive example is paired with a matched negative example. The negative is reconstructed by undoing the synthetic edit: for appended hallucinations the appended suffix is removed, and for replacement hallucinations the corrupted span is removed from the answer. This creates balanced evaluation sets:

| Dataset type | Positive | Negative | Total |
|---|---:|---:|---:|
| Contradictory factual hallucination | 736 | 736 | 1,472 |
| Overgeneration | 788 | 788 | 1,576 |
| Missing-tool violation | 788 | 788 | 1,576 |

## 2.2 Models and baselines

### Baseline 1: LettuceDetect

The first baseline is **LettuceDetect**, a pretrained ModernBERT-based hallucination detector. It is used zero-shot, without any additional training on the generated ToolACE hallucination data. The model receives the context, question, and answer, and returns predicted hallucinated character spans. This baseline is useful because it represents an existing hallucination detector trained for general RAG-style factuality detection rather than this specific tool-calling benchmark.

### Baseline 2: LookBackLens

The second baseline is **LookBackLens**. The intuition is that faithful tokens should attend strongly to the context, while hallucinated tokens may rely more on previously generated answer tokens. For each answer token, the method computes attention-based features from a causal language model. The feature for each layer and attention head is the ratio between attention to context tokens and attention to previous answer tokens.

Due to GPU memory constraints, the notebook uses a smaller Qwen backbone for this section instead of a 7B model. The attention features are extracted using eager attention, then a logistic regression classifier is trained over token-level labels. Predicted token probabilities are converted back into character spans.

### Proposed method: ModernBERT fine-tuning

The main improved model is a fine-tuned **ModernBERT token classifier**. The model is trained to classify tokens in the assistant answer using BIO labels:

| Label | Meaning |
|---|---|
| `O` | Non-hallucinated answer token |
| `B-HAL` | First token of a hallucinated span |
| `I-HAL` | Continuation token of a hallucinated span |

The input format is:

```text
QUESTION: <user query>

CONTEXT: <tool output>

ANSWER: <assistant answer>
```

Only answer tokens are supervised. Tokens belonging to the question, context, and special tokens receive label `-100`, so they do not contribute to the cross-entropy loss. This ensures that the model learns to localize hallucinated content in the answer rather than classify the entire prompt.

Positive and negative examples are kept as pairs during splitting to prevent leakage. The final split used in the notebook was:

| Split | Examples | Pairs |
|---|---:|---:|
| Train | 3,238 | 1,619 |
| Validation | 462 | 231 |
| Test | 924 | 462 |

After tokenization, a small number of examples were skipped because the answer was fully truncated. Evaluation was then run on the remaining encoded test examples.

The fine-tuning setup uses memory-safe training parameters suitable for a limited-GPU environment:

- per-device train batch size: 1;
- per-device evaluation batch size: 1;
- gradient accumulation steps: 8;
- gradient checkpointing: enabled;
- fp16 training when CUDA is available;
- best checkpoint selected by validation loss.

## 2.3 Evaluation metrics

The project evaluates predictions at two levels.

### Token/span-level evaluation

Predicted character spans and gold character spans are converted into binary character masks over the answer text. Precision, recall, and F1 are computed by aggregating true positive, false positive, and false negative characters across all positive examples.

This metric rewards exact localization. It is strict: if a method detects the correct example but marks too many surrounding characters, precision decreases.

### Example-level evaluation

Each example is converted into a binary decision: hallucinated or not hallucinated. A positive prediction is any non-empty predicted hallucinated span. Precision, recall, F1, and accuracy are computed on the balanced positive/negative evaluation set.

This metric evaluates whether a method detects the existence of hallucination in an answer, even if the predicted span boundary is imperfect.

## 2.4 Results

### Baseline results

The baseline comparison shows that LettuceDetect and LookBackLens perform well on appended hallucinations but struggle more with short contradictory replacements. LookBackLens generally improves token-level recall but often predicts hallucination for many examples, which hurts example-level precision and accuracy.

| Dataset | Method | Token P | Token R | Token F1 | Example P | Example R | Example F1 | Accuracy |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| Hallucination | LettuceDetect | 0.120 | 0.465 | 0.191 | 0.610 | 0.762 | 0.678 | 0.638 |
| Hallucination | LookBackLens | 0.196 | 0.717 | 0.308 | 0.488 | 1.000 | 0.656 | 0.492 |
| Overgeneration | LettuceDetect | 0.678 | 0.890 | 0.770 | 0.667 | 0.954 | 0.785 | 0.739 |
| Overgeneration | LookBackLens | 0.714 | 0.946 | 0.814 | 0.593 | 1.000 | 0.745 | 0.635 |
| Missing tool | LettuceDetect | 0.686 | 0.968 | 0.803 | 0.676 | 0.995 | 0.805 | 0.759 |
| Missing tool | LookBackLens | 0.842 | 0.964 | 0.899 | 0.622 | 1.000 | 0.767 | 0.677 |

### Final comparison with fine-tuned ModernBERT

Fine-tuned ModernBERT substantially improves both token-level and example-level results. The largest gain appears on short contradictory hallucinations, where the zero-shot baselines struggle to localize the exact corrupted substring.

| Dataset | Method | Token P | Token R | Token F1 | Example P | Example R | Example F1 | Accuracy |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| Hallucination | LettuceDetect | 0.120 | 0.465 | 0.191 | 0.610 | 0.762 | 0.678 | 0.638 |
| Hallucination | LookBackLens | 0.196 | 0.717 | 0.308 | 0.488 | 1.000 | 0.656 | 0.492 |
| Hallucination | ModernBERT-FT | **0.809** | 0.475 | **0.599** | **0.982** | 0.735 | **0.841** | **0.861** |
| Missing tool | LettuceDetect | 0.686 | 0.968 | 0.803 | 0.676 | 0.995 | 0.805 | 0.759 |
| Missing tool | LookBackLens | 0.842 | 0.964 | 0.899 | 0.622 | 1.000 | 0.767 | 0.677 |
| Missing tool | ModernBERT-FT | **0.997** | 0.962 | **0.979** | **0.947** | 0.973 | **0.959** | **0.959** |
| Overgeneration | LettuceDetect | 0.678 | 0.890 | 0.770 | 0.667 | 0.954 | 0.785 | 0.739 |
| Overgeneration | LookBackLens | 0.714 | 0.946 | 0.814 | 0.593 | 1.000 | 0.745 | 0.635 |
| Overgeneration | ModernBERT-FT | **0.998** | 0.952 | **0.975** | **0.945** | 0.951 | **0.948** | **0.948** |

## 2.5 Discussion

The results show three important trends.

First, contradictory factual hallucination is the hardest category. The corrupted span is often short, such as a number, status, name, or small phrase. A detector must localize a very narrow mismatch against the tool output. LettuceDetect reaches only 0.191 token F1 on this category, and LookBackLens reaches 0.308. Fine-tuned ModernBERT improves this to 0.599 token F1 and 0.841 example F1, mainly by greatly increasing precision.

Second, overgeneration is easier because the hallucinated region is usually a complete appended sentence. Both baselines perform reasonably well, but fine-tuning still gives a large improvement. ModernBERT-FT reaches 0.975 token F1 and 0.948 example F1.

Third, missing-tool violations are also detected well after fine-tuning. This category is not purely factual grounding; it depends on whether the assistant suggests an unavailable action. A general RAG hallucination detector can still flag many of these examples because the appended action is unsupported by the context, but the fine-tuned model performs best by learning the exact pattern of the task.

LookBackLens has very high recall but lower example-level precision. It often marks examples as positive, which explains the 1.000 example recall and lower accuracy. This behavior is useful for recall-oriented screening but less useful when precise decisions are needed.

## 2.6 Error analysis

The most common difficulty is span boundary precision. For short replacement hallucinations, the model may identify the correct area but include extra surrounding text. This lowers token-level precision even when example-level detection is correct.

Another challenge is semantic subtlety. Some contradictory replacements are small and natural-sounding, so the answer remains fluent. Detecting them requires comparing the answer against the exact tool output rather than relying on language-model plausibility.

For appended hallucinations, errors are more likely when the added sentence is highly related to the tool output. Such sentences can look like reasonable continuations, especially when the tool output provides partial evidence but not explicit support.

For missing-tool violations, the model must reason about the available tool list. If the answer suggests an action that sounds like a normal assistant follow-up, a general hallucination detector may not distinguish whether the required tool is available.

## 2.7 Reproducibility

The notebook is structured as a full pipeline:

1. install and configure dependencies;
2. define global settings and memory cleanup utilities;
3. check or generate datasets with `ensure_datasets()`;
4. build positive/negative evaluation pairs;
5. evaluate LettuceDetect;
6. extract LookBackLens features and train a logistic regression classifier;
7. fine-tune ModernBERT with BIO labels;
8. evaluate the fine-tuned model;
9. save result CSV files and the trained checkpoint.

The notebook also includes a bootstrap state cell that restores the main variables after a kernel restart. This is important because the pipeline uses large models and GPU memory has to be cleared between the generation, baseline, and fine-tuning stages.

The final output files are:

| File | Purpose |
|---|---|
| `toolace_halu_eval_results.csv` | Baseline results for LettuceDetect and LookBackLens |
| `toolace_halu_improve_results.csv` | Final comparison including ModernBERT-FT |
| `toolace_halu_modernbert_final/` | Saved fine-tuned model checkpoint |

## 2.8 Limitations

The generated dataset is synthetic. Although the examples are grounded in real ToolACE conversations, the hallucinations are injected by a model rather than collected from real user-facing failures. This makes the benchmark controlled and reproducible, but it may not cover all real-world hallucination patterns.

The negative examples are reconstructed by undoing the injected edit. This creates clean matched controls, but these negatives may be easier than naturally occurring non-hallucinated answers.

The LookBackLens implementation was adapted for limited GPU memory by using a smaller Qwen model. A larger backbone may produce stronger attention features but would require substantially more memory.

Finally, the current setup evaluates English-style tool outputs and assistant answers. Generalization to multilingual tool traces or more complex multi-tool workflows should be tested in future work.

## 2.9 Conclusion

This project builds a complete hallucination-detection pipeline for tool-calling conversations. Starting from ToolACE, it automatically constructs three RAGTruth-style hallucination datasets, evaluates two baselines, and fine-tunes a ModernBERT token classifier for span detection.

The fine-tuned model clearly outperforms the baselines across all hallucination categories. It is especially strong on overgeneration and missing-tool violations, reaching token F1 scores above 0.97. It also substantially improves contradictory factual hallucination detection, where the task is harder because the corrupted spans are short and require close comparison to the tool output.

Overall, the results show that task-specific fine-tuning is highly effective for hallucination detection in tool-calling settings, and that span-level supervision provides useful localization beyond simple binary hallucination classification.


In [ ]:
# Установка окружения. Выполняй эту ячейку первой.
import sys

!{sys.executable} -m pip install -q -U     vllm==0.6.3     datasets==2.21.0     huggingface_hub     scikit-learn     lettucedetect     lm-format-enforcer     accelerate     tqdm     pandas

# Важно: transformers ставим без зависимостей, чтобы не сломать стек vLLM.
!{sys.executable} -m pip install -q --force-reinstall --no-deps "transformers==4.46.3"

# Фикс для vLLM/outlines в некоторых окружениях.
!{sys.executable} -m pip install -q --force-reinstall "pyairports==2.1.1"


ERROR: Cannot install lettucedetect==0.1.0, lettucedetect==0.1.1, lettucedetect==0.1.2, lettucedetect==0.1.3, lettucedetect==0.1.4, lettucedetect==0.1.5, lettucedetect==0.1.6, lettucedetect==0.1.7, lettucedetect==0.1.8, vllm and vllm==0.6.3 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [22]:
import os, json, re, random, hashlib, gc, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset

# -------------------------
# Global config
# -------------------------
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/maramor/halu-datasets")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "hallucination":  DATA_DIR / "toolace_halu_contradiction.jsonl",
    "overgeneration": DATA_DIR / "toolace_halu_overgeneration.jsonl",
    "missing_tool":   DATA_DIR / "toolace_halu_missing_tool.jsonl",
}

# Generation config
GEN_MODEL = "Qwen/Qwen2.5-14B-Instruct"
N_SAMPLES = 1500              # per hallucination type
MAX_INPUT_TOKENS = 4096
GEN_BATCH_SIZE = 16

# Baseline / training config
TEST_FRAC = 0.30
MODEL_NAME = "answerdotai/ModernBERT-base"
OUTPUT_DIR = Path("./toolace_halu_modernbert_run")
VAL_FRAC = 0.10
FT_TEST_FRAC = 0.20
MAX_LEN = 1024
EPOCHS = 3
TRAIN_BSZ = 8
EVAL_BSZ = 16
LR = 3e-5


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    print("GPU/CPU memory cleanup done.")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


In [23]:
# -------------------------
# Dataset readiness helpers
# -------------------------
def jsonl_has_rows(path: Path, min_rows: int = 1) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        n = 0
        with path.open(encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    json.loads(line)
                    n += 1
                    if n >= min_rows:
                        return True
        return False
    except Exception as e:
        print(f"Broken JSONL: {path} -> {e}")
        return False


def datasets_ready(data_paths=DATASETS, min_rows: int = 1) -> bool:
    return all(jsonl_has_rows(path, min_rows=min_rows) for path in data_paths.values())


def dataset_status(data_paths=DATASETS):
    rows = []
    for name, path in data_paths.items():
        exists = path.exists()
        size = path.stat().st_size if exists else 0
        n_rows = None
        if exists and size > 0:
            try:
                n_rows = sum(1 for line in path.open(encoding="utf-8") if line.strip())
            except Exception:
                n_rows = "broken"
        rows.append({"dataset": name, "path": str(path), "exists": exists, "size_bytes": size, "rows": n_rows})
    return pd.DataFrame(rows)


def try_copy_existing_jsonl(source_dirs=(Path("."), Path("/mnt/data")), data_paths=DATASETS) -> bool:
    """If JSONL files are next to the notebook but not in DATA_DIR, copy them into DATA_DIR."""
    copied = []
    for target in data_paths.values():
        if jsonl_has_rows(target):
            continue
        for src_dir in source_dirs:
            src = src_dir / target.name
            if src.exists() and src.resolve() != target.resolve() and jsonl_has_rows(src):
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, target)
                copied.append((src, target))
                break
    if copied:
        for src, target in copied:
            print(f"Copied existing dataset: {src} -> {target}")
    return datasets_ready(data_paths)


def validate_dataset_spans(data_paths=DATASETS):
    report = []
    for name, path in data_paths.items():
        rows = load_jsonl(path)
        bad = 0
        for r in rows:
            for lbl in r.get("hallucination_labels", []):
                if r["output"][lbl["start"]:lbl["end"]] != lbl.get("text", ""):
                    bad += 1
        report.append({"dataset": name, "rows": len(rows), "span_mismatch": bad})
    return pd.DataFrame(report)


dataset_status()


,dataset,path,exists,size_bytes,rows
0,hallucination,/kaggle/input/datasets/maramor/halu-datasets/t...,True,905101,736
1,overgeneration,/kaggle/input/datasets/maramor/halu-datasets/t...,True,1176147,788
2,missing_tool,/kaggle/input/datasets/maramor/halu-datasets/t...,True,1181316,788


In [24]:
# -------------------------
# ToolACE parsing
# -------------------------

def _turn_role(t):
    return t.get("from") or t.get("role")


def _turn_text(t):
    return t.get("value") or t.get("content") or ""


def parse_record(rec):
    """Return (user_query, tools_text, tool_output, final_answer) or None."""
    convs = rec.get("conversations") or rec.get("messages") or []
    if not convs:
        return None

    tools_text = rec.get("system") or rec.get("tools") or ""
    if not tools_text:
        for t in convs:
            if _turn_role(t) == "system":
                tools_text = _turn_text(t)
                break

    user_query = None
    for t in convs:
        if _turn_role(t) == "user":
            user_query = _turn_text(t)
            break
    if not user_query:
        return None

    tool_idxs = [
        i for i, t in enumerate(convs)
        if _turn_role(t) in ("tool", "observation", "function")
    ]
    if not tool_idxs:
        return None

    last_tool = tool_idxs[-1]
    tool_output = _turn_text(convs[last_tool])

    final_answer = None
    for t in convs[last_tool + 1:]:
        if _turn_role(t) == "assistant":
            final_answer = _turn_text(t)
            break
    if not final_answer:
        return None

    if not final_answer.strip() or final_answer.strip().startswith("["):
        return None

    return (
        user_query.strip(),
        tools_text.strip(),
        tool_output.strip(),
        final_answer.strip(),
    )


def make_id(*parts):
    return hashlib.md5("||".join(parts).encode()).hexdigest()[:16]


def apply_replace(answer, original_span, corrupted_span):
    idx = answer.find(original_span)
    if idx < 0 or not corrupted_span or corrupted_span == original_span:
        return None

    new_answer = (
        answer[:idx]
        + corrupted_span
        + answer[idx + len(original_span):]
    )

    return new_answer, idx, idx + len(corrupted_span)


def apply_append(answer, added):
    if not added:
        return None

    added = added if added.startswith((" ", "\n")) else " " + added
    start = len(answer)
    new_answer = answer + added

    return new_answer, start, len(new_answer)


def to_ragtruth(rec_id, query, context, output, span_start, span_end, htype):
    return {
        "id": rec_id,
        "query": query,
        "context": context,
        "output": output,
        "hallucination_labels": [
            {
                "start": span_start,
                "end": span_end,
                "text": output[span_start:span_end],
                "type": htype,
            }
        ],
    }


def in_batches(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i + n]

In [25]:
# -------------------------
# Generation prompts / schemas
# -------------------------

SCHEMA_REPLACE = {
    "type": "object",
    "properties": {
        "original_span": {"type": "string", "minLength": 1},
        "corrupted_span": {"type": "string", "minLength": 1},
    },
    "required": ["original_span", "corrupted_span"],
}

SCHEMA_APPEND = {
    "type": "object",
    "properties": {
        "added_sentence": {"type": "string", "minLength": 5}
    },
    "required": ["added_sentence"],
}

SYS_HALU = (
    "You corrupt assistant answers for a hallucination-detection benchmark. "
    "Given a tool output (ground truth) and the assistant's answer, pick ONE short "
    "factual span in the answer that is directly supported by the tool output, and "
    "propose a replacement that CONTRADICTS the tool output. "
    "Keep the replacement the same surface form (number, word, name) so the sentence "
    "still reads naturally. Do not change anything else. "
    'Return JSON: {"original_span": "<exact substring of the answer>", '
    '"corrupted_span": "<replacement>"}. '
    "original_span MUST appear verbatim in the answer."
)

SYS_OVER = (
    "You corrupt assistant answers for a hallucination-detection benchmark. "
    "Given a tool output and the assistant's answer, write ONE additional sentence to "
    "append to the answer that introduces plausible information NOT present in the tool "
    "output (over-generation). The sentence must sound natural and topically related. "
    'Return JSON: {"added_sentence": "<one sentence, leading space included>"}.'
)

SYS_MISS = (
    "You corrupt assistant answers for a hallucination-detection benchmark. "
    "Given the LIST OF AVAILABLE TOOLS and the assistant's answer, write ONE additional "
    "sentence to append that offers the user a follow-up action that would REQUIRE A "
    "DIFFERENT TOOL not present in the available tools list, e.g. booking, ordering, "
    "emailing, scheduling — whichever is unrelated to the available tools. "
    'Return JSON: {"added_sentence": "<one sentence, leading space included>"}.'
)

In [27]:
# -------------------------
# Dataset generation
# -------------------------

def generate_datasets(
    data_paths=DATASETS,
    model_name=GEN_MODEL,
    n_samples=N_SAMPLES,
    batch_size=GEN_BATCH_SIZE,
    max_input_tokens=MAX_INPUT_TOKENS,
):
    """Generate the three RAGTruth-style ToolACE hallucination datasets."""
    from vllm import LLM, SamplingParams

    try:
        from vllm.sampling_params import GuidedDecodingParams
    except ImportError:
        try:
            from vllm import GuidedDecodingParams
        except ImportError:
            from vllm.model_executor.guided_decoding.guided_fields import GuidedDecodingParams

    print("Loading ToolACE...")
    ds = load_dataset("Team-ACE/ToolACE", split="train")

    parsed = []
    for rec in ds:
        p = parse_record(rec)
        if p is not None:
            parsed.append(p)

    print(f"Usable ToolACE records: {len(parsed)} / {len(ds)}")

    if not parsed:
        raise RuntimeError("No usable ToolACE records were parsed.")

    rng = random.Random(SEED)
    rng.shuffle(parsed)

    pool = parsed[: n_samples * 3 + 200]

    # Reuse full pool per type. ToolACE may have fewer usable records than 3*n_samples.
    slice_halu = pool[:n_samples]
    slice_over = pool[:n_samples]
    slice_miss = pool[:n_samples]

    print(
        f"Per-type slice size: "
        f"halu={len(slice_halu)} "
        f"over={len(slice_over)} "
        f"miss={len(slice_miss)}"
    )

    clear_gpu_memory()

    llm = LLM(
        model=model_name,
        dtype="bfloat16",
        max_model_len=max_input_tokens + 512,
        gpu_memory_utilization=0.90,
        enforce_eager=False,
        guided_decoding_backend="lm-format-enforcer",
    )

    tokenizer = llm.get_tokenizer()

    def chat(messages):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    def run_batch(prompts, schema, temperature=0.7, max_tokens=300):
        sp = SamplingParams(
            temperature=temperature,
            top_p=0.9,
            max_tokens=max_tokens,
            guided_decoding=GuidedDecodingParams(json=schema),
        )

        outs = llm.generate(prompts, sp, use_tqdm=True)

        parsed_out = []
        for o in outs:
            txt = o.outputs[0].text.strip()
            try:
                parsed_out.append(json.loads(txt))
            except Exception:
                parsed_out.append(None)

        return parsed_out

    def build_halu_prompt(tool_output, answer):
        user = f"""TOOL OUTPUT:
{tool_output}

ANSWER:
{answer}"""
        return chat([
            {"role": "system", "content": SYS_HALU},
            {"role": "user", "content": user},
        ])

    def build_over_prompt(tool_output, answer):
        user = f"""TOOL OUTPUT:
{tool_output}

ANSWER:
{answer}"""
        return chat([
            {"role": "system", "content": SYS_OVER},
            {"role": "user", "content": user},
        ])

    def build_miss_prompt(tools_text, answer):
        user = f"""AVAILABLE TOOLS:
{tools_text}

ANSWER:
{answer}"""
        return chat([
            {"role": "system", "content": SYS_MISS},
            {"role": "user", "content": user},
        ])

    try:
        # Type 1: contradiction / hallucination
        out_path = data_paths["hallucination"]
        kept = 0
        dropped = 0

        with out_path.open("w", encoding="utf-8") as f:
            for batch in in_batches(slice_halu, batch_size):
                prompts = [
                    build_halu_prompt(out_, ans)
                    for (_q, _tt, out_, ans) in batch
                ]

                results = run_batch(
                    prompts,
                    SCHEMA_REPLACE,
                    temperature=0.7,
                    max_tokens=200,
                )

                for (q, _tt, out_, ans), res in zip(batch, results):
                    if not res:
                        dropped += 1
                        continue

                    applied = apply_replace(
                        ans,
                        res.get("original_span", ""),
                        res.get("corrupted_span", ""),
                    )

                    if applied is None:
                        dropped += 1
                        continue

                    new_ans, s, e = applied

                    row = to_ragtruth(
                        make_id("halu", q, ans),
                        q,
                        out_,
                        new_ans,
                        s,
                        e,
                        "hallucination",
                    )

                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                    kept += 1

        print(f"hallucination: kept={kept} dropped={dropped} -> {out_path}")

        # Type 2: overgeneration
        out_path = data_paths["overgeneration"]
        kept = 0
        dropped = 0

        with out_path.open("w", encoding="utf-8") as f:
            for batch in in_batches(slice_over, batch_size):
                prompts = [
                    build_over_prompt(out_, ans)
                    for (_q, _tt, out_, ans) in batch
                ]

                results = run_batch(
                    prompts,
                    SCHEMA_APPEND,
                    temperature=0.8,
                    max_tokens=120,
                )

                for (q, _tt, out_, ans), res in zip(batch, results):
                    if not res or not res.get("added_sentence"):
                        dropped += 1
                        continue

                    applied = apply_append(ans, res["added_sentence"])

                    if applied is None:
                        dropped += 1
                        continue

                    new_ans, s, e = applied

                    row = to_ragtruth(
                        make_id("over", q, ans),
                        q,
                        out_,
                        new_ans,
                        s,
                        e,
                        "overgeneration",
                    )

                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                    kept += 1

        print(f"overgeneration: kept={kept} dropped={dropped} -> {out_path}")

        # Type 3: missing tool
        out_path = data_paths["missing_tool"]
        kept = 0
        dropped = 0

        with out_path.open("w", encoding="utf-8") as f:
            for batch in in_batches(slice_miss, batch_size):
                prompts = [
                    build_miss_prompt(tools, ans)
                    for (_q, tools, _out, ans) in batch
                ]

                results = run_batch(
                    prompts,
                    SCHEMA_APPEND,
                    temperature=0.8,
                    max_tokens=120,
                )

                for (q, _tt, out_, ans), res in zip(batch, results):
                    if not res or not res.get("added_sentence"):
                        dropped += 1
                        continue

                    applied = apply_append(ans, res["added_sentence"])

                    if applied is None:
                        dropped += 1
                        continue

                    new_ans, s, e = applied

                    row = to_ragtruth(
                        make_id("miss", q, ans),
                        q,
                        out_,
                        new_ans,
                        s,
                        e,
                        "missing_tool",
                    )

                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                    kept += 1

        print(f"missing_tool: kept={kept} dropped={dropped} -> {out_path}")

    finally:
        try:
            del llm
            del tokenizer
        except NameError:
            pass

        clear_gpu_memory()

    print("Validation report:")
    display(validate_dataset_spans(data_paths))

    return data_paths


def ensure_datasets(force_generate: bool = False, min_rows: int = 1):
    """Main entry point.

    - If all expected JSONL files exist and have rows: do nothing.
    - If files exist near the notebook but not inside DATA_DIR: copy them into DATA_DIR.
    - If DATA_DIR is empty/incomplete/broken: generate a fresh dataset.
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    print("Current dataset status:")
    display(dataset_status())

    if not force_generate and datasets_ready(min_rows=min_rows):
        print("Dataset is already ready. Generation is skipped.")
        display(validate_dataset_spans())
        return {
            name: load_jsonl(path)
            for name, path in DATASETS.items()
        }

    if not force_generate and try_copy_existing_jsonl():
        print("Dataset files were found outside DATA_DIR and copied. Generation is skipped.")
        display(validate_dataset_spans())
        return {
            name: load_jsonl(path)
            for name, path in DATASETS.items()
        }

    print("Dataset folder is empty/incomplete/broken. Starting generation...")
    generate_datasets(data_paths=DATASETS)

    if not datasets_ready(min_rows=min_rows):
        raise RuntimeError(
            "Dataset generation finished, but expected JSONL files are still missing or empty."
        )

    return {
        name: load_jsonl(path)
        for name, path in DATASETS.items()
    }

In [28]:
# Запусти эту ячейку перед baseline/evaluation/training.
# Если halu_datasets/ пустая, датасет будет сгенерирован автоматически.
data = ensure_datasets(force_generate=False, min_rows=1)
for name, rows in data.items():
    print(f"{name:>15}: {len(rows)} rows")


Current dataset status:


,dataset,path,exists,size_bytes,rows
0,hallucination,/kaggle/input/datasets/maramor/halu-datasets/t...,True,905101,736
1,overgeneration,/kaggle/input/datasets/maramor/halu-datasets/t...,True,1176147,788
2,missing_tool,/kaggle/input/datasets/maramor/halu-datasets/t...,True,1181316,788


Dataset is already ready. Generation is skipped.


,dataset,rows,span_mismatch
0,hallucination,736,0
1,overgeneration,788,0
2,missing_tool,788,0


  hallucination: 736 rows
 overgeneration: 788 rows
   missing_tool: 788 rows


## Baseline evaluation

Эта секция оценивает zero-shot LettuceDetect и LookBackLens. `data` уже загружен через `ensure_datasets()`.


## 3. Evaluation helpers

Methods predict hallucinated character ranges inside `output`. Ground truth comes from `hallucination_labels` (also character ranges). We score on two axes:

- **token-level** (binary mask over characters → P/R/F1): how well the predicted span overlaps the gold span.
- **example-level**: an example is "positive" if the gold set is non-empty (all our examples are positive); a prediction counts as TP if it flags any character of `output`. We add a *control* set of clean (un-edited) `output`s as negatives later for LettuceDetect.

In [29]:
import numpy as np

# -------------------------
# Metrics — robust version
# -------------------------

def char_mask(length, spans):
    """Binary mask over characters; 1 inside any span."""
    m = np.zeros(length, dtype=bool)

    if spans is None:
        return m

    if isinstance(spans, dict):
        if "spans" in spans:
            spans = spans["spans"]
        elif "predictions" in spans:
            spans = spans["predictions"]
        elif "start" in spans and "end" in spans:
            spans = [spans]
        else:
            return m

    if isinstance(spans, str):
        return m

    for s in spans:
        if not isinstance(s, dict):
            continue

        if "start" not in s or "end" not in s:
            continue

        try:
            a = max(0, int(s["start"]))
            b = min(length, int(s["end"]))
        except Exception:
            continue

        if b > a:
            m[a:b] = True

    return m


def token_level_prf(rows, pred_spans):
    """
    Character-level PRF.

    Supports both:
    1) pred_spans as list aligned with rows
    2) pred_spans as dict: {row_id: [{start, end}, ...]}
    """
    tp = 0
    fp = 0
    fn = 0

    for i, r in enumerate(rows):
        n = len(r["output"])

        gold = char_mask(
            n,
            r.get("hallucination_labels", []),
        )

        if isinstance(pred_spans, dict):
            pspans = pred_spans.get(r["id"], [])
        else:
            pspans = pred_spans[i]

        pred = char_mask(n, pspans)

        tp += int((gold & pred).sum())
        fp += int((~gold & pred).sum())
        fn += int((gold & ~pred).sum())

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

    return {
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def example_level_prf(rows, pred_spans, gold_positive_fn=None):
    """
    Binary detection per example.

    Supports both:
    1) pred_spans as list aligned with rows
    2) pred_spans as dict: {row_id: [{start, end}, ...]}
    """
    if gold_positive_fn is None:
        gold_positive_fn = lambda r: bool(r.get("hallucination_labels"))

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    for i, r in enumerate(rows):
        gold_pos = gold_positive_fn(r)

        if isinstance(pred_spans, dict):
            pspans = pred_spans.get(r["id"], [])
        else:
            pspans = pred_spans[i]

        pred_mask = char_mask(len(r["output"]), pspans)
        pred_pos = bool(pred_mask.any())

        if gold_pos and pred_pos:
            tp += 1
        elif gold_pos and not pred_pos:
            fn += 1
        elif not gold_pos and pred_pos:
            fp += 1
        else:
            tn += 1

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    acc = (tp + tn) / max(1, tp + tn + fp + fn)

    return {
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "accuracy": acc,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


# quick self-test
_row = {
    "id": "test",
    "output": "abcdefghij",
    "hallucination_labels": [{"start": 2, "end": 6}],
}

assert token_level_prf(
    [_row],
    {"test": [{"start": 4, "end": 8}]},
)["f1"] == 0.5

assert token_level_prf(
    [_row],
    [[{"start": 4, "end": 8}]],
)["f1"] == 0.5

print("eval helpers ready")

eval helpers ready


### Reconstruct the *clean* output for each row

To get a control set of negatives (for example-level eval) we need the **original** model answer — i.e. the `output` with the injected edit reverted. For TYPE 1 (replace) we revert the substring; for TYPE 2/3 (append) we trim the appended tail.

In [32]:
def clean_output(row):
    out = row["output"]
    labels = row.get("hallucination_labels") or []
    if not labels:
        return out
    lbl = labels[0]
    htype = lbl.get("type", "")
    if htype in ("overgeneration", "missing_tool"):
        return out[: lbl["start"]].rstrip()
    return out[: lbl["start"]] + out[lbl["end"]:]

def make_pair(row, halu_type):
    pos = {
        "id": row["id"],
        "query": row["query"],
        "context": row["context"],
        "output": row["output"],
        "hallucination_labels": row["hallucination_labels"],
        "type": halu_type,
        "is_positive": True,
    }
    neg = {
        "id": row["id"] + "_neg",
        "query": row["query"],
        "context": row["context"],
        "output": clean_output(row),
        "hallucination_labels": [],
        "type": "none",
        "is_positive": False,
        "pair_id": row["id"],
    }
    pos["pair_id"] = row["id"]
    return pos, neg

# Build a "balanced" eval set: original (negative) + corrupted (positive) for each row.
def build_balanced(rows):
    out = []
    for r in rows:
        out.append({**r, "is_positive": True})
        clean = clean_output(r)
        out.append({
            "id": r["id"] + "_neg",
            "query": r["query"],
            "context": r["context"],
            "output": clean,
            "hallucination_labels": [],
            "is_positive": False,
        })
    return out

balanced = {name: build_balanced(rows) for name, rows in data.items()}
for name, rows in balanced.items():
    pos = sum(1 for r in rows if r["is_positive"])
    print(f"{name:>15}: {len(rows)} examples ({pos} positive, {len(rows)-pos} negative)")

  hallucination: 1472 examples (736 positive, 736 negative)
 overgeneration: 1576 examples (788 positive, 788 negative)
   missing_tool: 1576 examples (788 positive, 788 negative)


In [33]:
# -------------------------
# Bootstrap / auto-restore notebook state
# -------------------------

def bootstrap_state(
    force_generate: bool = False,
    min_rows: int = 1,
    max_seq: int = 768,
):
    """
    Restores the main runtime variables after kernel restart.

    Creates:
    - datasets_by_type
    - data
    - balanced
    - MAX_SEQ
    """
    global datasets_by_type
    global data
    global balanced
    global MAX_SEQ

    # 1. Load or generate datasets
    datasets_by_type = ensure_datasets(
        force_generate=force_generate,
        min_rows=min_rows,
    )

    data = datasets_by_type
    MAX_SEQ = max_seq

    print("Loaded datasets:")
    for dtype, rows in datasets_by_type.items():
        print(f"  {dtype}: {len(rows)} examples")

    # 2. Build balanced positive/negative sets
    balanced = {}

    for name, rows in datasets_by_type.items():
        out = []

        for r in rows:
            pos, neg = make_pair(r, name)
            out.append(pos)
            out.append(neg)

        balanced[name] = out

    print("\nBalanced datasets:")
    for name, rows in balanced.items():
        n_pos = sum(1 for r in rows if r.get("is_positive"))
        n_neg = sum(1 for r in rows if not r.get("is_positive"))
        print(f"  {name}: total={len(rows)} pos={n_pos} neg={n_neg}")

    print(f"\nMAX_SEQ = {MAX_SEQ}")

    return {
        "datasets_by_type": datasets_by_type,
        "data": data,
        "balanced": balanced,
        "MAX_SEQ": MAX_SEQ,
    }


state = bootstrap_state(
    force_generate=False,
    min_rows=1,
    max_seq=768,
)

Current dataset status:


,dataset,path,exists,size_bytes,rows
0,hallucination,/kaggle/input/datasets/maramor/halu-datasets/t...,True,905101,736
1,overgeneration,/kaggle/input/datasets/maramor/halu-datasets/t...,True,1176147,788
2,missing_tool,/kaggle/input/datasets/maramor/halu-datasets/t...,True,1181316,788


Dataset is already ready. Generation is skipped.


,dataset,rows,span_mismatch
0,hallucination,736,0
1,overgeneration,788,0
2,missing_tool,788,0


Loaded datasets:
  hallucination: 736 examples
  overgeneration: 788 examples
  missing_tool: 788 examples

Balanced datasets:
  hallucination: total=1472 pos=736 neg=736
  overgeneration: total=1576 pos=788 neg=788
  missing_tool: total=1576 pos=788 neg=788

MAX_SEQ = 768


## 4. Baseline 1 — LettuceDetect

Pretrained ModernBERT-based span detector. We use the public checkpoint as-is (zero-shot for us — it was trained on RAGTruth which is human-judged Q/A hallucinations, not tool-calling traces, so this is a fair OOD baseline).

It returns predicted hallucinated spans inside `answer` given (`question`, `context`, `answer`) — we map directly to our (`query`, `context`, `output`).

In [34]:
# -------------------------
# Install LettuceDetect
# -------------------------
import sys

!{sys.executable} -m pip install -q -U lettucedetect

# Иногда lettucedetect подтягивает/ожидает совместимые версии transformers/torch.
# Если после установки будут ошибки уже внутри transformers, выполни:
# !{sys.executable} -m pip install -q -U "transformers>=4.46.0" accelerate safetensors

In [35]:
from lettucedetect.models.inference import HallucinationDetector

LETTUCE_MODEL = "KRLabsOrg/lettucedect-large-modernbert-en-v1"

lettuce = HallucinationDetector(
    method="transformer",
    model_path=LETTUCE_MODEL,
)
print(f"loaded {LETTUCE_MODEL}")

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

loaded KRLabsOrg/lettucedect-large-modernbert-en-v1


In [36]:
# -------------------------
# LettuceDetect helpers — robust version
# -------------------------

def normalize_lettuce_output(pred, answer):
    """
    Convert LettuceDetect output to:
    [
        {"start": int, "end": int, "text": str, "confidence": float},
        ...
    ]
    """
    if pred is None:
        return []

    # Case: {"spans": [...]} or {"predictions": [...]}
    if isinstance(pred, dict):
        if "spans" in pred:
            pred = pred["spans"]
        elif "predictions" in pred:
            pred = pred["predictions"]
        elif "start" in pred and "end" in pred:
            pred = [pred]
        else:
            return []

    # Case: string or unsupported object
    if isinstance(pred, str):
        return []

    if not isinstance(pred, list):
        return []

    spans = []

    for s in pred:
        if not isinstance(s, dict):
            continue

        start = s.get("start")
        end = s.get("end")

        if start is None or end is None:
            continue

        try:
            start = int(start)
            end = int(end)
        except Exception:
            continue

        if end <= start:
            continue

        start = max(0, start)
        end = min(len(answer), end)

        if end <= start:
            continue

        spans.append({
            "start": start,
            "end": end,
            "text": answer[start:end],
            "confidence": float(
                s.get(
                    "confidence",
                    s.get("score", s.get("hallucination_score", 1.0))
                )
            ),
        })

    return spans


def lettuce_predict_spans(row, detector=lettuce):
    question = row["query"]
    context = row["context"]
    answer = row["output"]

    if not isinstance(context, list):
        context = [context]

    pred = detector.predict(
        context=context,
        question=question,
        answer=answer,
        output_format="spans",
    )

    return normalize_lettuce_output(pred, answer)


def run_lettuce_on_rows(rows, detector=lettuce):
    pred_spans = {}

    for row in tqdm(rows, desc="LettuceDetect"):
        pred_spans[row["id"]] = lettuce_predict_spans(row, detector)

    return pred_spans

In [37]:
from tqdm.auto import tqdm

def lettuce_predict(rows):
    """Return list of [{start, end}, ...] aligned with rows."""
    preds = []
    for r in tqdm(rows, desc="lettuce", leave=False):
        try:
            out = lettuce.predict(
                context=[r["context"]],
                question=r["query"],
                answer=r["output"],
                output_format="spans",
            )
        except Exception as e:
            # malformed answer / empty → no prediction
            out = []
        spans = []
        for sp in out or []:
            # lettucedetect returns dicts with 'start','end' over the answer string
            if "start" in sp and "end" in sp and sp["end"] > sp["start"]:
                spans.append({"start": int(sp["start"]), "end": int(sp["end"])})
        preds.append(spans)
    return preds

# smoke test on 3 examples
demo = data["hallucination"][:3]
demo_pred = lettuce_predict(demo)
for r, p in zip(demo, demo_pred):
    print("GOLD:", r["hallucination_labels"])
    print("PRED:", p)
    print("OUT :", r["output"][:200], "...\n")

lettuce:   0%|          | 0/3 [00:00<?, ?it/s]

GOLD: [{'start': 20, 'end': 36, 'text': 'failed to remove', 'type': 'hallucination'}]
PRED: []
OUT : The player has been failed to remove from the sports program. ...

GOLD: [{'start': 109, 'end': 116, 'text': 'invalid', 'type': 'hallucination'}]
PRED: []
OUT : The verification results are in, and it appears that the email address wizardlywidgets@magicmail.com is both invalid and not disposable. You can confidently send your secrets to eternal youth to this  ...

GOLD: [{'start': 116, 'end': 122, 'text': 'Severe', 'type': 'hallucination'}]
PRED: [{'start': 77, 'end': 124}]
OUT : The impact analysis for the proposed structural change starting on 2023-08-01 has resulted in an overall rating of 'Severe'. Here are the recommendations to mitigate risks associated with the change:
 ...



In [38]:
# -------------------------
# Run LettuceDetect evaluation
# -------------------------

lettuce_results = []

for dtype, rows in datasets_by_type.items():
    print(f"=== LettuceDetect on {dtype} ({len(rows)} examples) ===")

    pred_spans = run_lettuce_on_rows(rows)

    predicted_positive = sum(
        1 for spans in pred_spans.values()
        if len(spans) > 0
    )

    print(f"Predicted positive examples: {predicted_positive} / {len(rows)}")

    tok = token_level_prf(rows, pred_spans)
    ex = example_level_prf(rows, pred_spans)

    print(
        f"  token-level   P={tok['precision']:.3f} "
        f"R={tok['recall']:.3f} "
        f"F1={tok['f1']:.3f}"
    )

    print(
        f"  example-level P={ex['precision']:.3f} "
        f"R={ex['recall']:.3f} "
        f"F1={ex['f1']:.3f} "
        f"acc={ex['accuracy']:.3f}"
    )

    lettuce_results.append({
        "method": "LettuceDetect",
        "dataset": dtype,
        "token_precision": tok["precision"],
        "token_recall": tok["recall"],
        "token_f1": tok["f1"],
        "example_precision": ex["precision"],
        "example_recall": ex["recall"],
        "example_f1": ex["f1"],
        "example_accuracy": ex["accuracy"],
        "predicted_positive": predicted_positive,
        "n_examples": len(rows),
    })

lettuce_df = pd.DataFrame(lettuce_results)
display(lettuce_df)

=== LettuceDetect on hallucination (736 examples) ===


LettuceDetect:   0%|          | 0/736 [00:00<?, ?it/s]

Predicted positive examples: 561 / 736
  token-level   P=0.120 R=0.465 F1=0.191
  example-level P=1.000 R=0.762 F1=0.865 acc=0.762
=== LettuceDetect on overgeneration (788 examples) ===


LettuceDetect:   0%|          | 0/788 [00:00<?, ?it/s]

Predicted positive examples: 752 / 788
  token-level   P=0.678 R=0.890 F1=0.770
  example-level P=1.000 R=0.954 F1=0.977 acc=0.954
=== LettuceDetect on missing_tool (788 examples) ===


LettuceDetect:   0%|          | 0/788 [00:00<?, ?it/s]

Predicted positive examples: 784 / 788
  token-level   P=0.686 R=0.968 F1=0.803
  example-level P=1.000 R=0.995 F1=0.997 acc=0.995


,method,dataset,token_precision,token_recall,token_f1,example_precision,example_recall,example_f1,example_accuracy,predicted_positive,n_examples
0,LettuceDetect,hallucination,0.120422,0.465471,0.191342,1.0,0.762228,0.865073,0.762228,561,736
1,LettuceDetect,overgeneration,0.678116,0.889771,0.769658,1.0,0.954315,0.976623,0.954315,752,788
2,LettuceDetect,missing_tool,0.686452,0.968423,0.803415,1.0,0.994924,0.997455,0.994924,784,788


In [39]:
lettuce_results = {}

for name, rows in balanced.items():
    print(f"=== LettuceDetect on {name} ({len(rows)} examples) ===")
    preds = lettuce_predict(rows)

    # token-level: score only on positives (negatives have no gold span by definition)
    pos_rows  = [r for r in rows if r["is_positive"]]
    pos_preds = [p for r, p in zip(rows, preds) if r["is_positive"]]
    tok = token_level_prf(pos_rows, pos_preds)

    # example-level on the balanced set
    ex = example_level_prf(rows, preds, gold_positive_fn=lambda r: r["is_positive"])

    lettuce_results[name] = {"token": tok, "example": ex, "preds": preds}
    print(f"  token-level   P={tok['precision']:.3f} R={tok['recall']:.3f} F1={tok['f1']:.3f}")
    print(f"  example-level P={ex['precision']:.3f} R={ex['recall']:.3f} F1={ex['f1']:.3f} acc={ex['accuracy']:.3f}")

=== LettuceDetect on hallucination (1472 examples) ===


lettuce:   0%|          | 0/1472 [00:00<?, ?it/s]

  token-level   P=0.120 R=0.465 F1=0.191
  example-level P=0.610 R=0.762 F1=0.678 acc=0.638
=== LettuceDetect on overgeneration (1576 examples) ===


lettuce:   0%|          | 0/1576 [00:00<?, ?it/s]

  token-level   P=0.678 R=0.890 F1=0.770
  example-level P=0.667 R=0.954 F1=0.785 acc=0.739
=== LettuceDetect on missing_tool (1576 examples) ===


lettuce:   0%|          | 0/1576 [00:00<?, ?it/s]

  token-level   P=0.686 R=0.968 F1=0.803
  example-level P=0.676 R=0.995 F1=0.805 acc=0.759


## 5. Baseline 2 — LookBackLens

LookBackLens (Chuang et al., 2024) classifies hallucinated generation by looking at how much attention each *generated* token paid to the **context** tokens versus the previously generated tokens.

Per token *t* in the answer, for every (layer *l*, head *h*) it computes

$$ \text{LBR}_{l,h}(t) = \frac{\sum_{c\in\text{context}} a_{l,h}(t,c)}{\sum_{c\in\text{context}} a_{l,h}(t,c) + \sum_{p<t,\,p\in\text{answer}} a_{l,h}(t,p)} $$

then a linear classifier over the per-(layer,head) means predicts hallucination at token / span / sentence level.

For tractable cost we:
- use a small but real LLM (`Qwen2.5-1.5B-Instruct`) — same family as the generator;
- extract per-token LBR features by re-running the *given* `output` in teacher-forced mode;
- pool features over the **gold span** vs **outside-span** tokens to get one (positive, negative) pair per example;
- fit a logistic regression and report token-level / example-level metrics under the same evaluation harness.

This deviates slightly from the paper (they slide a window over a freshly generated answer), but applied to our setting it's a faithful realization of "attention-to-context features → linear classifier".

In [40]:
# -------------------------
# Load LookBackLens backbone — memory-safe version
# -------------------------

import os
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Лучше задать до активного использования CUDA.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

clear_gpu_memory()

# На GPU ~15GB 7B + attentions обычно не помещается.
# Используем 1.5B как безопасный вариант для LookBackLens.
LBL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

lbl_tok = AutoTokenizer.from_pretrained(LBL_MODEL)

lbl_mdl = AutoModelForCausalLM.from_pretrained(
    LBL_MODEL,
    dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    attn_implementation="eager",  # нужно для output_attentions=True
    low_cpu_mem_usage=True,
).to(DEVICE).eval()

N_LAYERS = lbl_mdl.config.num_hidden_layers
N_HEADS = lbl_mdl.config.num_attention_heads
FEAT_DIM = N_LAYERS * N_HEADS

print(
    f"loaded {LBL_MODEL} on {DEVICE} | "
    f"L={N_LAYERS} H={N_HEADS} → feature dim {FEAT_DIM}"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-1.5B-Instruct on cuda | L=28 H=12 → feature dim 336


In [41]:
MAX_SEQ = 1024
datasets_by_type = ensure_datasets(force_generate=False, min_rows=1)
data = datasets_by_type

print("Loaded datasets:")
for dtype, rows in data.items():
    print(f"{dtype}: {len(rows)} examples")

Current dataset status:


,dataset,path,exists,size_bytes,rows
0,hallucination,/kaggle/input/datasets/maramor/halu-datasets/t...,True,905101,736
1,overgeneration,/kaggle/input/datasets/maramor/halu-datasets/t...,True,1176147,788
2,missing_tool,/kaggle/input/datasets/maramor/halu-datasets/t...,True,1181316,788


Dataset is already ready. Generation is skipped.


,dataset,rows,span_mismatch
0,hallucination,736,0
1,overgeneration,788,0
2,missing_tool,788,0


Loaded datasets:
hallucination: 736 examples
overgeneration: 788 examples
missing_tool: 788 examples


In [42]:
SYS_PROMPT_LBL = (
    "You answer the user's question using ONLY the provided tool output. "
    "Be concise and faithful to the tool output."
)

def build_chat_ids(row):
    """Encode (system, user(=query+context), assistant(=output)) as one sequence and
    return token ids plus the (start, end) indices that bound the assistant answer span
    and the context span — used to compute the lookback ratio."""
    user_msg = f"{row['query']}\n\nTOOL OUTPUT:\n{row['context']}"
    msgs_no_ans = [
        {"role": "system",    "content": SYS_PROMPT_LBL},
        {"role": "user",      "content": user_msg},
    ]
    prefix_text = lbl_tok.apply_chat_template(msgs_no_ans, tokenize=False, add_generation_prompt=True)
    full_text   = prefix_text + row["output"]

    prefix_ids = lbl_tok(prefix_text, add_special_tokens=False)["input_ids"]
    full_ids   = lbl_tok(full_text,   add_special_tokens=False)["input_ids"]

    # context span: from end-of-system to end of user msg — approximated as everything
    # before the assistant turn starts, after the first system block. We use prefix_ids
    # range as "non-answer" attention sources.
    ctx_start, ctx_end = 0, len(prefix_ids)
    ans_start, ans_end = len(prefix_ids), len(full_ids)
    return full_ids, (ctx_start, ctx_end), (ans_start, ans_end)

def map_char_spans_to_tokens(row, ans_start_tok):
    """Map gold character spans inside row['output'] to absolute token indices in the
    full encoded sequence. Returns a boolean mask of length (ans_end - ans_start)."""
    out = row["output"]
    enc = lbl_tok(out, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    mask = np.zeros(len(offsets), dtype=bool)
    for lbl in row.get("hallucination_labels", []):
        s, e = lbl["start"], lbl["end"]
        for i, (a, b) in enumerate(offsets):
            if b > s and a < e:
                mask[i] = True
    return mask  # length == answer token count

# quick sanity
_ids, _ctx, _ans = build_chat_ids(data["hallucination"][0])
print("seq len:", len(_ids), "| ctx:", _ctx, "| ans:", _ans, "| ans tokens:", _ans[1]-_ans[0])

seq len: 127 | ctx: (0, 115) | ans: (115, 127) | ans tokens: 12


In [43]:


@torch.no_grad()
def lookback_features(row):
    """Run one forward pass with output_attentions and return:
      feats:   (n_ans_tokens, N_LAYERS * N_HEADS)  per-token LBR features
      gold_mask: (n_ans_tokens,) bool
      ans_token_offsets: list[(char_start, char_end)] inside row['output']
    Returns None if the sequence is too long.
    """
    ids, (cs, ce), (as_, ae) = build_chat_ids(row)
    if len(ids) > MAX_SEQ or as_ >= ae:
        return None
    n_ans = ae - as_
    input_ids = torch.tensor([ids], device=DEVICE)
    out = lbl_mdl(input_ids, output_attentions=True, use_cache=False)

    eps = 1e-9
    # strict lower-triangular mask over the answer block, reused across layers
    tril = torch.tril(torch.ones(n_ans, n_ans, device=DEVICE, dtype=torch.float32), diagonal=-1)

    feats = torch.empty((n_ans, FEAT_DIM), dtype=torch.float32, device=DEVICE)
    for li, A in enumerate(out.attentions):
        # A: (1, heads, seq, seq) — only need the answer-row block
        ans_rows = A[0, :, as_:ae, :].to(torch.float32)        # (heads, n_ans, seq)
        ctx_attn  = ans_rows[:, :, cs:ce].sum(dim=-1)          # (heads, n_ans)
        prev_attn = (ans_rows[:, :, as_:ae] * tril).sum(dim=-1)  # (heads, n_ans)
        lbr = ctx_attn / (ctx_attn + prev_attn + eps)          # (heads, n_ans)
        feats[:, li * N_HEADS:(li + 1) * N_HEADS] = lbr.T      # (n_ans, heads)

    feats = feats.cpu().numpy()
    gold = map_char_spans_to_tokens(row, as_)
    L = min(len(gold), feats.shape[0])
    feats = feats[:L]
    gold = gold[:L]
    offsets = lbl_tok(row["output"], add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"][:L]
    return feats, gold, offsets

# smoke test
_r = data["overgeneration"][0]
_f = lookback_features(_r)
print("ok" if _f is None else (f"feats: {_f[0].shape}, gold positive tokens: {_f[1].sum()}/{len(_f[1])}"))

feats: (28, 336), gold positive tokens: 17/28


In [44]:
def extract_features_for_split(rows):
    """Return per-example feature dicts: {feats, gold, offsets} + a row pointer."""
    out = []
    for r in tqdm(rows, desc="lookback feats", leave=False):
        res = lookback_features(r)
        if res is None:
            continue
        feats, gold, offsets = res
        out.append({"row": r, "feats": feats, "gold": gold, "offsets": offsets})
    return out

def stack_for_classifier(examples):
    X = np.concatenate([e["feats"] for e in examples], axis=0)
    y = np.concatenate([e["gold"].astype(np.int8) for e in examples], axis=0)
    return X, y

# Train/test split per dataset; we only fit on positives (negatives have no positive
# tokens but provide many neg tokens too, which is fine).
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

lookback_results = {}

def predict_spans_from_token_probs(probs, offsets, threshold=0.5):
    """probs: per-token P(hallucination). Convert to char-level spans by merging
    consecutive positive tokens."""
    spans, cur = [], None
    for p, (a, b) in zip(probs, offsets):
        if p >= threshold and b > a:
            if cur is None: cur = [a, b]
            else: cur[1] = b
        else:
            if cur is not None:
                spans.append({"start": cur[0], "end": cur[1]})
                cur = None
    if cur is not None:
        spans.append({"start": cur[0], "end": cur[1]})
    return spans

for name, rows in balanced.items():
    print(f"=== LookBackLens on {name} ===")

    # split BEFORE feature extraction to keep eval clean
    rng = random.Random(SEED)
    indices = list(range(len(rows)))
    rng.shuffle(indices)
    n_test = int(len(rows) * TEST_FRAC)
    test_idx = set(indices[:n_test])
    train_rows = [rows[i] for i in indices[n_test:]]
    test_rows  = [rows[i] for i in indices[:n_test]]

    train_examples = extract_features_for_split(train_rows)
    test_examples  = extract_features_for_split(test_rows)
    if not train_examples or not test_examples:
        print("  not enough examples after filtering — skipping")
        continue

    X_tr, y_tr = stack_for_classifier(train_examples)
    X_te, y_te = stack_for_classifier(test_examples)
    print(f"  train tokens: {len(y_tr)} (pos {int(y_tr.sum())}) | test tokens: {len(y_te)} (pos {int(y_te.sum())})")

    scaler = StandardScaler().fit(X_tr)
    clf = LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=-1)
    clf.fit(scaler.transform(X_tr), y_tr)

    # token-level classification report (just for diagnostics)
    y_pred_tok = clf.predict(scaler.transform(X_te))
    print(classification_report(y_te, y_pred_tok, digits=3, zero_division=0))

    # reconstruct spans per test example and score with our harness
    pred_spans_per_row = []
    test_row_seq = []
    for ex in test_examples:
        probs = clf.predict_proba(scaler.transform(ex["feats"]))[:, 1]
        spans = predict_spans_from_token_probs(probs, ex["offsets"], threshold=0.5)
        pred_spans_per_row.append(spans)
        test_row_seq.append(ex["row"])

    pos_rows  = [r for r in test_row_seq if r["is_positive"]]
    pos_preds = [p for r, p in zip(test_row_seq, pred_spans_per_row) if r["is_positive"]]
    tok = token_level_prf(pos_rows, pos_preds)
    ex_m = example_level_prf(test_row_seq, pred_spans_per_row, gold_positive_fn=lambda r: r["is_positive"])

    lookback_results[name] = {"token": tok, "example": ex_m}
    print(f"  token-level   P={tok['precision']:.3f} R={tok['recall']:.3f} F1={tok['f1']:.3f}")
    print(f"  example-level P={ex_m['precision']:.3f} R={ex_m['recall']:.3f} F1={ex_m['f1']:.3f} acc={ex_m['accuracy']:.3f}")

=== LookBackLens on hallucination ===


lookback feats:   0%|          | 0/1031 [00:00<?, ?it/s]

lookback feats:   0%|          | 0/441 [00:00<?, ?it/s]

  train tokens: 103625 (pos 3735) | test tokens: 47140 (pos 1478)
              precision    recall  f1-score   support

           0      0.992     0.818     0.897     45662
           1      0.125     0.799     0.216      1478

    accuracy                          0.818     47140
   macro avg      0.558     0.809     0.556     47140
weighted avg      0.965     0.818     0.876     47140

  token-level   P=0.196 R=0.717 F1=0.308
  example-level P=0.488 R=1.000 F1=0.656 acc=0.492
=== LookBackLens on overgeneration ===


lookback feats:   0%|          | 0/1104 [00:00<?, ?it/s]

lookback feats:   0%|          | 0/472 [00:00<?, ?it/s]

  train tokens: 129461 (pos 12715) | test tokens: 56843 (pos 5896)
              precision    recall  f1-score   support

           0      0.992     0.909     0.949     50947
           1      0.544     0.934     0.688      5896

    accuracy                          0.912     56843
   macro avg      0.768     0.922     0.818     56843
weighted avg      0.945     0.912     0.922     56843

  token-level   P=0.714 R=0.946 F1=0.814
  example-level P=0.593 R=1.000 F1=0.745 acc=0.635
=== LookBackLens on missing_tool ===


lookback feats:   0%|          | 0/1104 [00:00<?, ?it/s]

lookback feats:   0%|          | 0/472 [00:00<?, ?it/s]

  train tokens: 130670 (pos 13787) | test tokens: 57626 (pos 6679)
              precision    recall  f1-score   support

           0      0.994     0.954     0.974     50947
           1      0.734     0.959     0.831      6679

    accuracy                          0.955     57626
   macro avg      0.864     0.957     0.903     57626
weighted avg      0.964     0.955     0.957     57626

  token-level   P=0.842 R=0.964 F1=0.899
  example-level P=0.622 R=1.000 F1=0.767 acc=0.677


## 6. Summary table

In [45]:
import pandas as pd

rows = []
for ds in DATASETS:
    for method_name, method_res in [("LettuceDetect", lettuce_results), ("LookBackLens", lookback_results)]:
        r = method_res.get(ds)
        if r is None:
            continue
        rows.append({
            "dataset": ds,
            "method": method_name,
            "tok_P":  round(r["token"]["precision"], 3),
            "tok_R":  round(r["token"]["recall"], 3),
            "tok_F1": round(r["token"]["f1"], 3),
            "ex_P":   round(r["example"]["precision"], 3),
            "ex_R":   round(r["example"]["recall"], 3),
            "ex_F1":  round(r["example"]["f1"], 3),
            "ex_acc": round(r["example"]["accuracy"], 3),
        })
df = pd.DataFrame(rows)
df.to_csv("toolace_halu_eval_results.csv", index=False)
df

,dataset,method,tok_P,tok_R,tok_F1,ex_P,ex_R,ex_F1,ex_acc
0,hallucination,LettuceDetect,0.120,0.465,0.191,0.610,0.762,0.678,0.638
1,hallucination,LookBackLens,0.196,0.717,0.308,0.488,1.000,0.656,0.492
2,overgeneration,LettuceDetect,0.678,0.890,0.770,0.667,0.954,0.785,0.739
3,overgeneration,LookBackLens,0.714,0.946,0.814,0.593,1.000,0.745,0.635
4,missing_tool,LettuceDetect,0.686,0.968,0.803,0.676,0.995,0.805,0.759
5,missing_tool,LookBackLens,0.842,0.964,0.899,0.622,1.000,0.767,0.677


## Fine-tuning ModernBERT и финальное сравнение

Эта секция обучает ModernBERT на паре positive/negative examples, считает span-level и example-level метрики, затем объединяет результат с baseline CSV.


In [46]:
# Reuse datasets loaded by ensure_datasets().
# The baseline section used TEST_FRAC=0.30 for LookBackLens.
# For fine-tuning we switch back to the intended ModernBERT split.
TEST_FRAC = FT_TEST_FRAC
raw = data
for name, rows in raw.items():
    print(f"{name:>15}: {len(rows)}")


  hallucination: 736
 overgeneration: 788
   missing_tool: 788


### Build positives + matched negatives, then split

Negatives are reconstructed by undoing the injected edit (same logic as in `toolace_halu_eval.ipynb`). Each (positive, negative) pair stays together in the same split to avoid leakage.

In [48]:


# Build the unified pool: one entry = one (pos, neg) pair, keyed by source row.
pairs = []
for name, rows in raw.items():
    for r in rows:
        pairs.append(make_pair(r, name))

random.Random(SEED).shuffle(pairs)
n_total = len(pairs)
n_test  = int(n_total * TEST_FRAC)
n_val   = int(n_total * VAL_FRAC)
test_pairs  = pairs[:n_test]
val_pairs   = pairs[n_test:n_test + n_val]
train_pairs = pairs[n_test + n_val:]

def flatten(ps):
    out = []
    for pos, neg in ps:
        out.append(pos); out.append(neg)
    return out

train_ex = flatten(train_pairs)
val_ex   = flatten(val_pairs)
test_ex  = flatten(test_pairs)

print(f"train: {len(train_ex)} (pairs: {len(train_pairs)})")
print(f"val:   {len(val_ex)} (pairs: {len(val_pairs)})")
print(f"test:  {len(test_ex)} (pairs: {len(test_pairs)})")

# Per-type counts in test (sanity)
from collections import Counter
print("test by type:", Counter(e["type"] for e in test_ex))

train: 3238 (pairs: 1619)
val:   462 (pairs: 231)
test:  924 (pairs: 462)
test by type: Counter({'none': 462, 'overgeneration': 163, 'hallucination': 151, 'missing_tool': 148})


## 2. Tokenize + align BIO labels over the answer span only

Strategy: encode each example as `query [SEP] context [SEP] answer`. Label only the subword tokens that fall inside the *answer* segment — everything in `query`/`context`/specials gets label `-100` so cross-entropy ignores it. Inside the answer, tokens that overlap any gold span get `B-HAL` (first) or `I-HAL` (continuation); other answer tokens get `O`.

In [49]:
from transformers import AutoTokenizer

LABELS = ["O", "B-HAL", "I-HAL"]
L2I = {l: i for i, l in enumerate(LABELS)}
I2L = {i: l for l, i in L2I.items()}
IGNORE_INDEX = -100

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode_example(ex):
    """Return a dict with input_ids, attention_mask, labels and offset_mapping (for eval)."""
    q, c, a = ex["query"], ex["context"], ex["output"]
    # Build the raw text deterministically so we can locate the answer offset in chars.
    prefix = f"QUESTION: {q}\n\nCONTEXT: {c}\n\nANSWER: "
    full = prefix + a
    ans_char_start = len(prefix)

    enc = tokenizer(
        full,
        truncation=True,
        max_length=MAX_LEN,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )
    offsets = enc.pop("offset_mapping")

    # Build gold char mask over the WHOLE input string
    gold_char_mask = np.zeros(len(full), dtype=bool)
    for lbl in ex.get("hallucination_labels", []):
        s = ans_char_start + lbl["start"]
        e = ans_char_start + lbl["end"]
        if e > s:
            gold_char_mask[max(0, s): min(len(full), e)] = True

    labels = []
    prev_was_pos = False
    for (a_off, b_off) in offsets:
        if a_off == b_off == 0:
            # special token
            labels.append(IGNORE_INDEX)
            prev_was_pos = False
            continue
        if b_off <= ans_char_start:
            # in question/context prefix → don't supervise
            labels.append(IGNORE_INDEX)
            prev_was_pos = False
            continue
        # answer token
        token_is_pos = gold_char_mask[a_off:b_off].any()
        if token_is_pos:
            labels.append(L2I["I-HAL"] if prev_was_pos else L2I["B-HAL"])
            prev_was_pos = True
        else:
            labels.append(L2I["O"])
            prev_was_pos = False

    enc["labels"] = labels
    # keep offsets + ans_char_start for evaluation later (we'll pop them out of the dataset before training)
    enc["__offsets"] = offsets
    enc["__ans_char_start"] = ans_char_start
    enc["__full_len"] = len(full)
    return enc

# smoke test
_e = encode_example(train_ex[0])
print("len:", len(_e["input_ids"]), "pos labels:", sum(1 for l in _e["labels"] if l in (1, 2)))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

len: 360 pos labels: 31


In [50]:
from tqdm.auto import tqdm

def encode_split(split):
    encoded = []
    skipped = 0
    for ex in tqdm(split, desc="encode", leave=False):
        enc = encode_example(ex)
        # if the answer was fully truncated, skip
        if not any(l != IGNORE_INDEX for l in enc["labels"]):
            skipped += 1
            continue
        # carry source row pointer for eval
        enc["__src"] = ex
        encoded.append(enc)
    print(f"skipped {skipped} (no answer tokens left after truncation)")
    return encoded

train_enc = encode_split(train_ex)
val_enc   = encode_split(val_ex)
test_enc  = encode_split(test_ex)
print(f"encoded: train={len(train_enc)} val={len(val_enc)} test={len(test_enc)}")

encode:   0%|          | 0/3238 [00:00<?, ?it/s]

skipped 18 (no answer tokens left after truncation)


encode:   0%|          | 0/462 [00:00<?, ?it/s]

skipped 6 (no answer tokens left after truncation)


encode:   0%|          | 0/924 [00:00<?, ?it/s]

skipped 6 (no answer tokens left after truncation)
encoded: train=3220 val=456 test=918


## 3. Train ModernBERT token classifier

`Trainer` over a tiny torch `Dataset`. We strip the `__*` private keys before training, then put them back at eval time.

In [52]:
# -------------------------
# Free GPU before ModernBERT fine-tuning
# -------------------------

import gc
import torch

for name in ["lbl_mdl", "lbl_tok", "lettuce", "trainer", "model"]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

print("GPU memory cleared")

GPU memory cleared


In [54]:
# -------------------------
# Train ModernBERT token classifier — memory-safe version
# -------------------------

import os
import gc
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

class HaluDataset(Dataset):
    """Wrap a list of encoded examples; expose only model-required keys."""
    KEEP = ("input_ids", "attention_mask", "labels")

    def __init__(self, encoded):
        self.encoded = encoded

    def __len__(self):
        return len(self.encoded)

    def __getitem__(self, i):
        e = self.encoded[i]
        return {
            k: e[k]
            for k in self.KEEP
            if k in e
        }


train_ds = HaluDataset(train_enc)
val_ds = HaluDataset(val_enc)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=I2L,
    label2id=L2I,
)

model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    label_pad_token_id=IGNORE_INDEX,
)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=EPOCHS,
    learning_rate=LR,

    # memory-safe
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # use fp16 on 15GB GPUs; bf16 can be heavier/unsupported depending on hardware
    fp16=torch.cuda.is_available(),
    bf16=False,

    dataloader_num_workers=0,
    logging_steps=25,
    report_to=[],
    seed=SEED,
)

try:
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        tokenizer=tokenizer,
    )

trainer.train()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
 

Epoch,Training Loss,Validation Loss
1,1.204447,0.118739
2,0.556662,0.075340
3,0.139255,0.062710


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=606, training_loss=1.0422892401320707, metrics={'train_runtime': 1789.0506, 'train_samples_per_second': 5.4, 'train_steps_per_second': 0.339, 'total_flos': 2526924629500884.0, 'train_loss': 1.0422892401320707, 'epoch': 3.0})

## 4. Evaluation

Same harness as `toolace_halu_eval.ipynb` (char-level P/R/F1 + example-level P/R/F1). We predict per-token labels, then reconstruct character spans inside the *answer* using the original token offsets (subtracting the answer prefix offset).

In [55]:
def char_mask(length, spans):
    m = np.zeros(length, dtype=bool)
    for s in spans:
        a, b = max(0, s["start"]), min(length, s["end"])
        if b > a:
            m[a:b] = True
    return m

def token_level_prf(rows, pred_spans):
    tp = fp = fn = 0
    for r, pspans in zip(rows, pred_spans):
        n = len(r["output"])
        gold = char_mask(n, r["hallucination_labels"])
        pred = char_mask(n, pspans)
        tp += int((gold & pred).sum())
        fp += int((~gold & pred).sum())
        fn += int((gold & ~pred).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {"precision": prec, "recall": rec, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

def example_level_prf(rows, pred_spans, gold_positive_fn=None):
    if gold_positive_fn is None:
        gold_positive_fn = lambda r: bool(r.get("hallucination_labels"))
    tp = fp = fn = tn = 0
    for r, pspans in zip(rows, pred_spans):
        gold_pos = gold_positive_fn(r)
        pred_pos = any((s["end"] > s["start"]) for s in pspans)
        if   gold_pos and pred_pos: tp += 1
        elif gold_pos and not pred_pos: fn += 1
        elif not gold_pos and pred_pos: fp += 1
        else: tn += 1
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    acc  = (tp + tn) / max(1, tp + tn + fp + fn)
    return {"precision": prec, "recall": rec, "f1": f1, "accuracy": acc,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}

In [56]:
@torch.no_grad()
def predict_one(enc):
    """Run the trained model on a single pre-encoded example and return per-token
    predicted labels aligned with enc['__offsets']."""
    device = next(model.parameters()).device
    ids  = torch.tensor([enc["input_ids"]],     device=device)
    attn = torch.tensor([enc["attention_mask"]], device=device)
    logits = model(input_ids=ids, attention_mask=attn).logits[0].float().cpu().numpy()
    pred_ids = logits.argmax(-1)
    return pred_ids

def spans_from_pred(pred_ids, offsets, ans_char_start):
    """Convert token-level predictions to character spans within row['output']."""
    spans, cur = [], None
    for pid, (a, b) in zip(pred_ids, offsets):
        if a == b == 0 or b <= ans_char_start:
            # special / prefix → flush
            if cur is not None:
                spans.append({"start": cur[0], "end": cur[1]}); cur = None
            continue
        rel_a = max(0, a - ans_char_start)
        rel_b = b - ans_char_start
        label = I2L[int(pid)]
        if label in ("B-HAL", "I-HAL"):
            if cur is None or label == "B-HAL":
                if cur is not None:
                    spans.append({"start": cur[0], "end": cur[1]})
                cur = [rel_a, rel_b]
            else:
                cur[1] = rel_b
        else:
            if cur is not None:
                spans.append({"start": cur[0], "end": cur[1]}); cur = None
    if cur is not None:
        spans.append({"start": cur[0], "end": cur[1]})
    return spans

model.eval()
pred_spans_all = []
rows_all = []
for enc in tqdm(test_enc, desc="predict test"):
    pred_ids = predict_one(enc)
    spans = spans_from_pred(pred_ids, enc["__offsets"], enc["__ans_char_start"])
    pred_spans_all.append(spans)
    rows_all.append(enc["__src"])
print("done")

predict test:   0%|          | 0/918 [00:00<?, ?it/s]

done


In [57]:
import pandas as pd

# Per-type results: hallucination / overgeneration / missing_tool — score positives of
# that type alongside their negatives.
modernbert_results = {}

for halu_type in ["hallucination", "overgeneration", "missing_tool"]:
    pos_idx = [i for i, r in enumerate(rows_all) if r["type"] == halu_type and r["is_positive"]]
    # matched negatives = the neg from the same pair (have type == "none" and pair_id matches)
    pos_pair_ids = {rows_all[i]["pair_id"] for i in pos_idx}
    neg_idx = [i for i, r in enumerate(rows_all)
               if not r["is_positive"] and r.get("pair_id") in pos_pair_ids]
    idx = pos_idx + neg_idx
    rows  = [rows_all[i]       for i in idx]
    preds = [pred_spans_all[i] for i in idx]
    pos_rows  = [r for r in rows if r["is_positive"]]
    pos_preds = [p for r, p in zip(rows, preds) if r["is_positive"]]
    tok = token_level_prf(pos_rows, pos_preds)
    ex  = example_level_prf(rows, preds, gold_positive_fn=lambda r: r["is_positive"])
    modernbert_results[halu_type] = {"token": tok, "example": ex, "n_pos": len(pos_idx), "n_neg": len(neg_idx)}
    print(f"{halu_type:>15}: n_pos={len(pos_idx)} n_neg={len(neg_idx)} | "
          f"tok F1={tok['f1']:.3f} (P={tok['precision']:.3f} R={tok['recall']:.3f}) | "
          f"ex F1={ex['f1']:.3f} (P={ex['precision']:.3f} R={ex['recall']:.3f} acc={ex['accuracy']:.3f})")

  hallucination: n_pos=151 n_neg=151 | tok F1=0.599 (P=0.809 R=0.475) | ex F1=0.841 (P=0.982 R=0.735 acc=0.861)
 overgeneration: n_pos=162 n_neg=162 | tok F1=0.975 (P=0.998 R=0.952) | ex F1=0.948 (P=0.945 R=0.951 acc=0.948)
   missing_tool: n_pos=146 n_neg=146 | tok F1=0.979 (P=0.997 R=0.962) | ex F1=0.959 (P=0.947 R=0.973 acc=0.959)


## 5. Comparison with baselines

Load baseline numbers from `toolace_halu_eval_results.csv` and add our model's row.

In [58]:
baseline_csv = Path("toolace_halu_eval_results.csv")
if baseline_csv.exists():
    baselines = pd.read_csv(baseline_csv)
else:
    print("WARN: baseline CSV not found — run toolace_halu_eval.ipynb first to populate it.")
    baselines = pd.DataFrame(columns=["dataset", "method", "tok_P", "tok_R", "tok_F1",
                                      "ex_P", "ex_R", "ex_F1", "ex_acc"])

ours_rows = []
for halu_type, r in modernbert_results.items():
    ours_rows.append({
        "dataset": halu_type,
        "method": "ModernBERT-FT (ours)",
        "tok_P":  round(r["token"]["precision"], 3),
        "tok_R":  round(r["token"]["recall"], 3),
        "tok_F1": round(r["token"]["f1"], 3),
        "ex_P":   round(r["example"]["precision"], 3),
        "ex_R":   round(r["example"]["recall"], 3),
        "ex_F1":  round(r["example"]["f1"], 3),
        "ex_acc": round(r["example"]["accuracy"], 3),
    })
ours_df = pd.DataFrame(ours_rows)

combined = pd.concat([baselines, ours_df], ignore_index=True)
combined = combined.sort_values(["dataset", "method"]).reset_index(drop=True)
combined.to_csv("toolace_halu_improve_results.csv", index=False)
combined

,dataset,method,tok_P,tok_R,tok_F1,ex_P,ex_R,ex_F1,ex_acc
0,hallucination,LettuceDetect,0.120,0.465,0.191,0.610,0.762,0.678,0.638
1,hallucination,LookBackLens,0.196,0.717,0.308,0.488,1.000,0.656,0.492
2,hallucination,ModernBERT-FT (ours),0.809,0.475,0.599,0.982,0.735,0.841,0.861
3,missing_tool,LettuceDetect,0.686,0.968,0.803,0.676,0.995,0.805,0.759
4,missing_tool,LookBackLens,0.842,0.964,0.899,0.622,1.000,0.767,0.677
5,missing_tool,ModernBERT-FT (ours),0.997,0.962,0.979,0.947,0.973,0.959,0.959
6,overgeneration,LettuceDetect,0.678,0.890,0.770,0.667,0.954,0.785,0.739
7,overgeneration,LookBackLens,0.714,0.946,0.814,0.593,1.000,0.745,0.635
8,overgeneration,ModernBERT-FT (ours),0.998,0.952,0.975,0.945,0.951,0.948,0.948


## 6. (optional) Save trained model

Saves the fine-tuned checkpoint locally — uncomment the `push_to_hub` block to publish.

In [62]:
SAVE_DIR = Path("./toolace_halu_modernbert_final")
trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))
print("saved to", SAVE_DIR)

from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(folder_path=".", repo_id="Fawnnn/ModernBERT-toolace-hallu-detect", repo_type="model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to toolace_halu_modernbert_final


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Fawnnn/ModernBERT-toolace-hallu-detect/commit/4faa39d3f56427332b2f81333d2c8d8c7effd624', commit_message='Upload folder using huggingface_hub', commit_description='', oid='4faa39d3f56427332b2f81333d2c8d8c7effd624', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Fawnnn/ModernBERT-toolace-hallu-detect', endpoint='https://huggingface.co', repo_type='model', repo_id='Fawnnn/ModernBERT-toolace-hallu-detect'), pr_revision=None, pr_num=None)